# xAI MTS: free Colab real-update pilot

無料GPUで、既存の固定済み Qwen2.5-0.5B / GSM8K パイロットを実行します。ランタイムの種類を GPU にして、このノートブックを対話的に実行してください。GPUが割り当てられないときは停止します。Colab Proの購入や有料API呼び出しは行いません。

3 seeds × trusted/shuffled reward、16 train / 16 eval tasks、各4更新、FP32の既存プロトコルを変更せず使います。正負・ゼロの結果と失敗を残します。これは小規模パイロットであり、MLSysタスクでの能力向上、長時間エージェント、複数GPUスケーリングの証拠にはなりません。

ソースは `30b3a438ac7bd77a0cc216ff250873a9b8818925` に固定し、実行スクリプトは元の設定に従って別の systems SHA を使います。元の研究プロトコルとシステムの2つの版が保存されます。

In [ ]:
from pathlib import Path
import json, subprocess, sys, datetime
SOURCE_SHA = "30b3a438ac7bd77a0cc216ff250873a9b8818925"
ATTEMPT = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CHECKOUT = Path("/content") / ("rvl-pilot-" + ATTEMPT)
subprocess.run(["git", "clone", "https://github.com/mitukx/Recursive-Verification-Lag.git", str(CHECKOUT)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_SHA], cwd=CHECKOUT, check=True)
lock = json.loads((CHECKOUT / "configs/colab_rlvr_pilot_v1.json").read_text())
print(json.dumps(lock, indent=2))


## GPUとメモリの確認

単一GPUの実験です。無料枠の割り当て・接続時間は保証されません。割り当て不可やメモリ不足を、別モデルや別精度への自動変更で隠しません。変更する場合は新しい実験ロックを作ります。

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPUが未割り当てです。ランタイムをGPUに変更するか、無料枠の別の機会に再試行してください。")
print(subprocess.check_output(["nvidia-smi"], text=True))
print("torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## 最小依存関係

Colab付属のCUDA対応PyTorchを維持し、元のプロトコルで固定された Transformers / Datasets / Accelerate のみをインストールします。vLLM、Docker、分散ワーカーはこのパイロットでは使いません。環境は後で保存します。

In [ ]:
deps = [f"{name}=={version}" for name, version in lock["dependencies"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", *deps], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)


## 実行と失敗の保存

stdout/stderrを表示しながら保存します。評価データで学習更新やハイパーパラメータを選びません。既存ランナーはCUDAなし、greedy再現性の失敗、OOM等で停止し、可能な限り途中の結果と失敗を保存します。途中でセッションが切れた場合も、次の書き出しセルから部分結果をダウンロードしてください。

In [ ]:
EVIDENCE = CHECKOUT / "notebook-evidence"
EVIDENCE.mkdir(exist_ok=False)
(EVIDENCE / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
(EVIDENCE / "notebook-source.json").write_text(json.dumps({"source_sha": SOURCE_SHA, "attempt": ATTEMPT, "paid_compute_started": False}, indent=2))
with (EVIDENCE / "runner.log").open("w") as log:
    process = subprocess.Popen([sys.executable, "-u", "scripts/run_colab_rlvr_pilot.py"], cwd=CHECKOUT,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    returncode = process.wait()
(EVIDENCE / "exit.json").write_text(json.dumps({"returncode": returncode}, indent=2))
print("exit code:", returncode, "（0以外も削除せず保持します）")


## 結果の確認とダウンロード

`summary.json`、各seed/armのterminal予測、rollout、history、zero-update control、環境、失敗、checksumをまとめます。成功したファイルだけを選別しません。精度差がゼロや負でもそのまま報告してください。16問・4更新からMTS採用や一般的な能力向上は主張できません。

In [ ]:
import hashlib, shutil
RESULT = CHECKOUT / "results/colab_rlvr_pilot_v1"
if RESULT.exists():
    shutil.copytree(RESULT, EVIDENCE / "pilot-results", dirs_exist_ok=True)
    summary = RESULT / "summary.json"
    if summary.exists():
        print(summary.read_text())
manifest = {str(p.relative_to(EVIDENCE)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in EVIDENCE.rglob("*") if p.is_file() and p.name != "bundle-manifest.json"}
(EVIDENCE / "bundle-manifest.json").write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(str(Path("/content") / ("rvl-pilot-evidence-" + ATTEMPT)), "zip", EVIDENCE)
print("保存した結果:", archive)
from google.colab import files
files.download(archive)
